In [8]:
import pandas as pd
import numpy as np
import os
import gseapy as gp
from Utils import compute_jaccard_indices, extract_significances, spearman_across_datasets
from GSEA import plot_pathway



seed = 3
# Load origin DGE once (used for every seed)
original_dge = pd.read_csv(f'../DGE/DGE_Del9p21/Seed_{seed}/DGE_Origin.csv')

# Helper: safe status filtering (tolerant to missing Status column)
def get_safe_df(df, status_col='Status'):
    """
    Small internal helper (script-local): return rows with status=='ok' if the status column exists,
    otherwise return a shallow copy of the dataframe.
    """
    if status_col in df.columns:
        return df[df[status_col] == 'ok'].copy()
    return df.copy()


# for seed in seeds:
    # Build dges_results per-seed (do NOT accumulate across seeds)
dges_results = {}
dges_results['Origin'] = original_dge.copy()

synthetic_datasets = [
    f"avatarsk5_{seed}",
    f"avatarsk10_{seed}",
    f"ctgan_{seed}",
    f"gaussiancopula_{seed}",
    f"synthpop_{seed}",
    f"tvae_{seed}",
]

# Load synthetic DGE results for this seed (skip missing files)
for synthetic_dataset in synthetic_datasets:
    path = f'../DGE/DGE_Del9p21/Seed_{seed}/DGE_{synthetic_dataset}.csv'
    try:
        dge_syn_result = pd.read_csv(path)
        dges_results[synthetic_dataset] = dge_syn_result
    except Exception as err:
        print(f"Skipping {synthetic_dataset}: could not read {path}: {err}")

Skipping tvae_3: could not read ../DGE/DGE_Del9p21/Seed_3/DGE_tvae_3.csv: [Errno 2] No such file or directory: '../DGE/DGE_Del9p21/Seed_3/DGE_tvae_3.csv'


In [9]:
import gseapy as gp
hallmarks_gmt = "h.all.v2025.1.Hs.symbols.gmt"
GSEA_overall = {}
for name, degs in dges_results.items():
    print(f"----GSEA {name}----")
    safe_df = get_safe_df(degs)
    safe_df['Rank_Score'] = np.sign(safe_df['Log2FC'])*(-np.log10(safe_df['P_value']))
    ranked_results_df = safe_df.sort_values(by="Rank_Score", ascending = False)
    rnk = ranked_results_df[["Gene","Rank_Score"]].set_index("Gene")
    pre_res = gp.prerank(rnk=rnk,
                         gene_sets=hallmarks_gmt,
                         threads=32,
                         permutation_num=10000, # reduce number to speed up testing
                         outdir=None, # don't write to disk
                         seed=seed,
                         verbose=True)
    prerank_overall = pre_res.res2d
    prerank_overall.to_csv(f"9p21/Seed_{seed}/GSEA_{name}_{seed}.csv",index = False)
    GSEA_overall[name] = prerank_overall

----GSEA Origin----


2026-01-07 18:40:34,273 [WARNING] Duplicated values found in preranked stats: 83.23% of genes
The order of those genes will be arbitrary, which may produce unexpected results.
2026-01-07 18:40:34,275 [INFO] Parsing data files for GSEA.............................
2026-01-07 18:40:34,376 [INFO] 0000 gene_sets have been filtered out when max_size=500 and min_size=15
2026-01-07 18:40:34,379 [INFO] 0050 gene_sets used for further statistical testing.....
2026-01-07 18:40:34,380 [INFO] Start to run GSEA...Might take a while..................
2026-01-07 18:40:59,553 [INFO] Congratulations. GSEApy runs successfully................

2026-01-07 18:40:59,596 [WARNING] Duplicated values found in preranked stats: 86.42% of genes
The order of those genes will be arbitrary, which may produce unexpected results.
2026-01-07 18:40:59,597 [INFO] Parsing data files for GSEA.............................
2026-01-07 18:40:59,652 [INFO] 0000 gene_sets have been filtered out when max_size=500 and min_size=15


----GSEA avatarsk5_3----


2026-01-07 18:41:24,934 [INFO] Congratulations. GSEApy runs successfully................

2026-01-07 18:41:24,986 [WARNING] Duplicated values found in preranked stats: 86.91% of genes
The order of those genes will be arbitrary, which may produce unexpected results.
2026-01-07 18:41:24,988 [INFO] Parsing data files for GSEA.............................
2026-01-07 18:41:25,044 [INFO] 0000 gene_sets have been filtered out when max_size=500 and min_size=15
2026-01-07 18:41:25,045 [INFO] 0050 gene_sets used for further statistical testing.....
2026-01-07 18:41:25,046 [INFO] Start to run GSEA...Might take a while..................


----GSEA avatarsk10_3----


...

2026-01-07 18:42:15,459 [INFO] Congratulations. GSEApy runs successfully................

2026-01-07 18:42:15,554 [WARNING] Duplicated values found in preranked stats: 88.06% of genes
The order of those genes will be arbitrary, which may produce unexpected results.
2026-01-07 18:42:15,555 [INFO] Parsing data files for GSEA.............................
2026-01-07 18:42:15,617 [INFO] 0000 gene_sets have been filtered out when max_size=500 and min_size=15
2026-01-07 18:42:15,618 [INFO] 0050 gene_sets used for further statistical testing.....
2026-01-07 18:42:15,619 [INFO] Start to run GSEA...Might take a while..................


----GSEA gaussiancopula_3----


2026-01-07 18:42:40,310 [INFO] Congratulations. GSEApy runs successfully................

2026-01-07 18:42:40,389 [WARNING] Duplicated values found in preranked stats: 89.08% of genes
The order of those genes will be arbitrary, which may produce unexpected results.
2026-01-07 18:42:40,391 [INFO] Parsing data files for GSEA.............................
2026-01-07 18:42:40,448 [INFO] 0000 gene_sets have been filtered out when max_size=500 and min_size=15
2026-01-07 18:42:40,449 [INFO] 0050 gene_sets used for further statistical testing.....
2026-01-07 18:42:40,450 [INFO] Start to run GSEA...Might take a while..................


----GSEA synthpop_3----


2026-01-07 18:43:05,297 [INFO] Congratulations. GSEApy runs successfully................

